Access data programmatically from the Data Ware House
=========================

Get a personal access token:

1. Got to [https://tableau.epfl.ch](https://tableau.epfl.ch)
2. Open your account settings by clicking on your initials on the top right:
<img src="doc/img/tableau_account_settings.png" width=800/>
3. Scroll down to the "Personal Access Tokens" section:
<img src="doc/img/tableau_create_personal_access_token.png" width=800/>
4. Enter the name for the new token.
5. Click on the "Create Token"button.
6. Copy the secret and the token name.
<img src="doc/img/tableau_view_personal_access_token.png" width=800/>
7. create a configuration file `conf/pat-prod.env` with the following info:

```properties
SERVER_URL='https://tableau.epfl.ch'
ACCESS_TOKEN_NAME='TOKEN_NAME_FROM_PREVIOUS_STEP'
PERSONAL_ACCESS_TOKEN='TOKEN_SECRET_FROM_PREVIOUS_STEP'
```

Load the configuration file, authenticate and initialize the VizQL client using this configuration:

In [ ]:
from dotenv import dotenv_values
import tableauserverclient as TSC
from vizql_data_service_py import VizQLDataServiceClient

# Load the settings
settings = dotenv_values('conf/pat-prod.env')
# define the server and authentication method
tableau_auth = TSC.PersonalAccessTokenAuth(settings['ACCESS_TOKEN_NAME'], settings['PERSONAL_ACCESS_TOKEN'])
server = TSC.Server(settings['SERVER_URL'])
# Authenticate
server.auth.sign_in(tableau_auth)
# Initialize the VizQL client
client = VizQLDataServiceClient(settings['SERVER_URL'], server, tableau_auth)

1. Go to [https://tableau.epfl.ch](https://tableau.epfl.ch)
2. Click on "Explore" in the left panel
3. Find the datasource you want and open it by clicking on it
4. Open the datasource details by clicking on the "<b>ⓘ</b>" on the top right, next to the datasource name.
5. The LUID is displayed at the bottom of the datasource details panel:

![Tableau's datasource details panel](doc/img/tableau_get_luid.png)

Alternatively you can get information about all the datasources (including the LUID) with:

In [ ]:
from json import dumps
from utils import obj_to_dict

# 1. Get info (including the LUID) of all datasources
data_sources = [obj_to_dict(ds) for ds in TSC.Pager(server.datasources)]
print('List of data_sources:', dumps(data_sources, indent=4))

Put the LUID of the datasource in the code bellow and initialize the datasource for VizQL:

In [ ]:
from vizql_data_service_py import Datasource

datasource = Datasource(datasourceLuid='11c8c5a5-95ed-434c-bd27-83c71e705e91')

You can now get the metadata of the datasource which describe each field with its name, caption, datatype and possibly the source.

In [ ]:
from json import dumps
from vizql_data_service_py import ReadMetadataRequest, read_metadata

# create the metadata request and retrieve the response
read_metadata_request = ReadMetadataRequest(datasource=datasource)
read_metadata_response = read_metadata.sync(client=client, body=read_metadata_request)

# simplify the response and display the metadata
metadata = [md.model_dump(mode='json', exclude_unset=True) for md in read_metadata_response.data]
print(f"Fields description: {dumps(metadata, indent=4)}")

From the list of fields, define the ones you want to download.

In [ ]:
from vizql_data_service_py import DimensionField, Query

# list the fields manually
fields_to_get = [
    DimensionField(fieldCaption='Infoscience ID'),
    DimensionField(fieldCaption='Doc Title'),
    DimensionField(fieldCaption='JOURNAL'),
]

# or get all fields:
#fields_to_get = [DimensionField(fieldCaption=md.fieldCaption) for md in read_metadata_response.data]

Now query the API to download the data for the list of fields you selected:

In [ ]:
from vizql_data_service_py import Query, QueryRequest, query_datasource, QueryDatasourceOptions

# create the data query and run it to get the data
# currently a bug prevent limiting the number of rows with options=QueryDatasourceOptions(rowLimit=100)
query_request = QueryRequest(query=Query(fields=fields_to_get), datasource=datasource)
query_response = query_datasource.sync(client=client, body=query_request)

if query_response is None:
    # if there is an error getting the data, print some details
    print(query_datasource.sync_detailed(client=client, body=query_request))
else:
    # show the first 100 rows for the selected fields
    print(f"Data: {dumps(query_response.data[:100], indent=4)}")

It is also possible to use aggregation (see the [visql library documentation](https://github.com/tableau/VizQL-Data-Service/blob/main/python_sdk/README.md)):

In [ ]:
from vizql_data_service_py import Function, MeasureField, SortDirection

# count the number of publication for each journal and order by the number of publications
fields_agg = [
    DimensionField(fieldCaption='JOURNAL'),
    MeasureField(
        fieldAlias='#Pub', fieldCaption='Infoscience ID',
        function=Function.COUNTD, sortPriority=1, sortDirection=SortDirection.DESC,
    ),
]
query_request = QueryRequest(query=Query(fields=fields_agg), datasource=datasource)
query_response = query_datasource.sync(client=client, body=query_request)

if query_response is None:
    # if there is an error getting the data, print some details
    print(query_datasource.sync_detailed(client=client, body=query_request))
else:
    # show the first 100 rows for the selected fields
    print(f"Data: {dumps(query_response.data[:100], indent=4)}")